# 🎙️ Speech-to-Text Model Router (Colab + Gradio)

Pick an open-source STT model from a dropdown → it is set up automatically in **its own venv** → upload or record audio → get the transcript. Switch models any time without restarting the runtime.

**How it works**
- The notebook kernel runs only **Gradio + the router**. It never imports torch/transformers.
- Each model gets a venv in `/content/envs/<model>` and runs in a separate **worker process** that talks to the router with JSON lines over stdin/stdout.
- Switching models **stops the old worker first** (frees the GPU), then starts the new one.
- Venvs use `--system-site-packages`, so they reuse Colab's CUDA torch and only install extras. A model that needs its own torch sets `isolated_torch: true`.
- Weights load from fast local disk. A backup of each model (one `.tar`) is kept on Google Drive and restored after a runtime reset, so big downloads happen once.
- If cached weights turn out to be damaged, the router deletes them, downloads again and retries automatically.

**Run order:** Runtime → *Change runtime type* → **T4 GPU**, then run cells top to bottom. Step 7 opens the UI.

**Gated models** (Indic-Transcribe-Flex, IndicConformer): open the model page on Hugging Face, accept the terms, then paste your token in the **Hugging Face token** box at the top of the UI (or add a Colab secret named `HF_TOKEN`). The box tells you which gated models your account can access.

## 1 · Settings, folders, Drive backup and Hugging Face token

In [ ]:
# ==== Settings ===============================================================
USE_DRIVE = True   # keep a backup of downloaded weights on Google Drive (one .tar per model)
SHARE     = False  # True = public gradio.live link. Keep False: some model licences
                   # (e.g. Indic-Transcribe) need permission to host for other people.
# =============================================================================
import os, pathlib

for d in ["/content/stt_router/adapters", "/content/stt_router/logs",
          "/content/envs", "/content/models", "/content/in", "/content/hf_cache"]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# Weights are always LOADED from fast local disk. Drive is only a backup that gets
# restored to local disk on a fresh runtime (loading straight from the Drive mount
# can give corrupted reads such as "SafetensorError: header too large").
os.environ["HF_HOME"] = "/content/hf_cache"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["STT_DRIVE_CACHE"] = "/content/drive/MyDrive/stt_router/weights_backup"
    os.makedirs(os.environ["STT_DRIVE_CACHE"], exist_ok=True)
    old = pathlib.Path("/content/drive/MyDrive/stt_router/hf_cache")
    if old.exists():
        print(f"Note: {old} is from the previous version and is no longer used - delete it to free Drive space.")
else:
    os.environ.pop("STT_DRIVE_CACHE", None)

# Optional: a Colab secret named HF_TOKEN. You can also paste the token in the UI.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    print("No HF_TOKEN secret - paste your token in the UI (top of the page) for gated models.")
print("HF cache:", os.environ["HF_HOME"], "| Drive backup:", os.environ.get("STT_DRIVE_CACHE", "off"))

## 2 · Main-process libraries
Only the router's own needs. Model libraries go into each model's venv, not here.

In [ ]:
!pip -q install -U gradio
!pip -q install pyyaml uv jiwer
!ffmpeg -version | head -n 1
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"

## 3 · Model registry
One entry per model. **To add a model:** add an entry here + one adapter file (step 4), then press *Reload registry.yaml* in the UI's System tab (or re-run step 6).

Starting set, picked from your model list: a tiny Whisper for smoke tests, a strong Hindi Whisper, Indic-Transcribe-Flex, IndicConformer 600M, and MMS-1B-all. Pin `revision` to a commit once a model works.

In [ ]:
%%writefile /content/stt_router/registry.yaml
# ---------------------------------------------------------------
# STT Model Router - model registry
# One entry per model. To add a model: add an entry here and an
# adapter file in adapters/. Nothing else changes.
#
# Fields
#   display        Label shown in the Gradio dropdown
#   source         hf (Hugging Face Hub) or github
#   repo           HF repo id, or git URL when source: github
#   revision       Branch, tag or commit. Pin a commit once a model works.
#   prefetch       true  -> setup manager downloads the full repo first
#                           (needed when the model ships its own code)
#                  false -> the worker lets transformers fetch weights
#   gated          true if you must accept terms on the HF model page
#   deps           Extra pip packages for this model's venv
#                  Do NOT list torch / torchaudio here unless
#                  isolated_torch is true: they come from Colab.
#   isolated_torch true -> venv does NOT see Colab's torch; list a torch
#                  version in deps and it is installed fresh with uv
#   languages      Codes shown in the language dropdown ("auto" = let
#                  the model detect it, only if the model supports it)
#   adapter        File name in adapters/ (without .py)
#   options        Fixed settings passed to the adapter's load()
#   ui_options     Per-request choices shown in the UI (e.g. output mode)
# ---------------------------------------------------------------
models:

  whisper-small-vaani-hindi:
    display: "Whisper small | Vaani Hindi (ARTPARK-IISc) - quick smoke test"
    source: hf
    repo: ARTPARK-IISc/whisper-small-vaani-hindi
    revision: main
    prefetch: false
    gated: false
    deps: [transformers, soundfile]
    isolated_torch: false
    languages: [hi]
    adapter: whisper_adapter

  whisper-hindi-large-v2:
    display: "Whisper large-v2 | Hindi (vasista22, IIT Madras Speech Lab)"
    source: hf
    repo: vasista22/whisper-hindi-large-v2
    revision: main
    prefetch: false
    gated: false
    deps: [transformers, soundfile]
    isolated_torch: false
    languages: [hi]
    adapter: whisper_adapter

  indic-transcribe-flex:
    display: "Indic-Transcribe-Flex | 27 languages (Bodhan AI) - gated"
    source: hf
    repo: bodhan-ai/indic-transcribe-flex
    revision: main
    prefetch: true          # model code ships inside the repo
    gated: true
    deps: [transformers, sentencepiece, soundfile]
    isolated_torch: false
    languages: [auto, hi, en, bn, ta, te, mr, gu, kn, ml, pa, or, as, ur, ne,
                sa, mai, kok, doi, brx, ks, mni, sat, sd, bho, hne, bgc, bhb]
    adapter: indic_transcribe_adapter
    ui_options:
      mode: [native, mixed, romanized]

  indic-conformer-600m:
    display: "IndicConformer 600M | 22 languages (AI4Bharat) - gated"
    source: hf
    repo: ai4bharat/indic-conformer-600m-multilingual
    revision: main
    prefetch: false
    gated: true
    # Pinned per the model card (onnx / onnxruntime 1.20.x). Newer onnxruntime
    # rejects this model's external weight files when they sit in the HF cache
    # ("External data path escapes model directory"). The card says 1.20.1, but
    # onnxruntime-gpu 1.20.1 has no Python 3.13 wheel, so use 1.20.2.
    # Only ONE onnxruntime package per venv: the -gpu build also has the CPU provider.
    deps: [transformers, soundfile, "onnx==1.20.1", "onnxruntime-gpu==1.20.2"]
    isolated_torch: false
    languages: [hi, bn, ta, te, mr, gu, kn, ml, pa, or, as, ur, ne, sa,
                mai, kok, doi, brx, ks, mni, sat, sd]
    adapter: indic_conformer_adapter
    ui_options:
      decoding: [ctc, rnnt]

  mms-1b-all:
    display: "MMS-1B-all | 1000+ languages (Meta) - non-commercial licence"
    source: hf
    repo: facebook/mms-1b-all
    revision: main
    prefetch: false         # full repo has ~1000 adapter files; fetch on demand
    gated: false
    deps: [transformers, soundfile]
    isolated_torch: false
    languages: [hi, bn, ta, te, mr, gu, kn, ml, pa, or, as, ur, ne, en]
    adapter: mms_adapter

## 4 · Adapters
Each adapter implements `load()` and `transcribe()`. They run inside the model's venv, so they can import whatever that venv has.

In [ ]:
%%writefile /content/stt_router/adapters/base.py
"""Common adapter interface. Every model adapter subclasses BaseAdapter."""


class BaseAdapter:
    def load(self, model_ref: str, cfg: dict) -> None:
        """Load weights (onto the GPU if there is one). Called once when the worker starts.
        model_ref is a local folder (prefetch: true) or a Hugging Face repo id."""
        raise NotImplementedError

    def transcribe(self, audio_path: str, lang: str | None = None, **options) -> dict:
        """audio_path is always a 16 kHz mono WAV. Return {"text": str, ...extra fields}."""
        raise NotImplementedError


def device():
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"


def read_wav(path):
    """16 kHz mono float32 numpy array (the router already converted the file)."""
    import soundfile as sf
    audio, sr = sf.read(path, dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    assert sr == 16000, f"expected 16 kHz, got {sr}"
    return audio


def need_lang(lang, model):
    if not lang or lang == "auto":
        raise ValueError(f"{model} needs a language - pick one in the Language dropdown")

In [ ]:
%%writefile /content/stt_router/adapters/whisper_adapter.py
"""Any Whisper fine-tune that works with the transformers ASR pipeline."""
import sys

import torch
from transformers import pipeline

from base import BaseAdapter, read_wav


class Adapter(BaseAdapter):
    def load(self, model_ref, cfg):
        cuda = torch.cuda.is_available()
        self.pipe = pipeline(
            "automatic-speech-recognition",
            model=model_ref,
            revision=cfg.get("revision"),
            torch_dtype=torch.float16 if cuda else torch.float32,
            device=0 if cuda else -1,
        )

    def transcribe(self, audio_path, lang=None, **options):
        audio = {"raw": read_wav(audio_path), "sampling_rate": 16000}
        kw = {"task": "transcribe"}
        if lang and lang != "auto":
            kw["language"] = lang
        try:
            out = self.pipe(dict(audio), chunk_length_s=30, batch_size=8, generate_kwargs=kw)
        except Exception as e:  # older fine-tunes reject language/task kwargs
            print(f"[whisper] retrying without generate_kwargs: {e}", file=sys.stderr)
            out = self.pipe(dict(audio), chunk_length_s=30, batch_size=8)
        return {"text": out["text"].strip()}

In [ ]:
%%writefile /content/stt_router/adapters/indic_transcribe_adapter.py
"""Bodhan AI Indic-Transcribe-Flex. Code ships inside the HF repo (prefetch: true). Gated.
Licence note: hosting it as a service for OTHER people needs Bodhan AI's written permission."""
import sys

import soundfile as sf

from base import BaseAdapter


class Adapter(BaseAdapter):
    def load(self, model_ref, cfg):
        sys.path.insert(0, model_ref)  # model_ref is the downloaded repo folder
        from indic_transcribe import IndicTranscribe
        self.asr = IndicTranscribe.from_pretrained(model_ref)
        try:
            from long_form import transcribe_long
            self.transcribe_long = transcribe_long
        except ImportError:
            self.transcribe_long = None

    def transcribe(self, audio_path, lang=None, mode="native", **options):
        kw = {"mode": mode or "native"}
        if lang and lang != "auto":
            kw["lang"] = lang
        if sf.info(audio_path).duration > 30:
            if self.transcribe_long is None:
                raise RuntimeError("Audio > 30 s and long_form.py was not found in the model repo")
            return {"text": self.transcribe_long(self.asr, audio_path, **kw)}
        if "lang" not in kw:  # auto: also report the detected language if the API allows it
            try:
                text, lid = self.asr.transcribe(audio_path, return_lid=True, mode=kw["mode"])
                return {"text": text, "detected_lang": lid.get("lang")}
            except TypeError:
                pass
        return {"text": self.asr(audio_path, **kw)}

In [ ]:
%%writefile /content/stt_router/adapters/indic_conformer_adapter.py
"""AI4Bharat IndicConformer 600M multilingual (remote code, ONNX, CTC or RNNT decoding). Gated.

ONNX + Hugging Face cache gotcha: the cache keeps real files in blobs/<hash> and puts symlinks
in snapshots/<rev>/. Newer onnxruntime checks that a model's external weight files live inside
the model's directory, and the symlinks can make them look like they "escape" it. So before
onnxruntime opens a model from the cache, we rebuild that folder from hard links (real files,
same directory, no extra disk space) and open it from there. Versions are also pinned per the
model card in registry.yaml.
"""
import hashlib
import os
import shutil
import sys
from pathlib import Path

import torch  # imported first so its CUDA libraries are loaded for onnxruntime-gpu
from transformers import AutoModel

from base import BaseAdapter, need_lang, read_wav

REAL_ROOT = Path(os.environ.get("HF_HOME", "/content/hf_cache")) / "onnx_real"


def materialize(path):
    """Same ONNX file, but inside a folder of real files (hard links to the cache blobs)."""
    p = Path(path)
    if not p.is_symlink():
        return path
    src_dir = p.parent
    dst_dir = REAL_ROOT / hashlib.sha1(str(src_dir).encode()).hexdigest()[:12]
    if not (dst_dir / ".done").exists():
        for f in src_dir.rglob("*"):
            if f.is_dir():
                continue
            t = dst_dir / f.relative_to(src_dir)
            t.parent.mkdir(parents=True, exist_ok=True)
            t.unlink(missing_ok=True)
            real = f.resolve()
            try:
                os.link(real, t)            # instant, no extra space (same filesystem)
            except OSError:
                shutil.copy2(real, t)       # different filesystem: fall back to a copy
        (dst_dir / ".done").touch()
        print(f"[indic-conformer] real-file copy of {src_dir} -> {dst_dir}", file=sys.stderr)
    return str(dst_dir / p.name)


def patch_onnxruntime(ort):
    if getattr(ort.InferenceSession, "_stt_patched", False):
        return
    base = ort.InferenceSession

    class Session(base):
        _stt_patched = True

        def __init__(self, path_or_bytes, *args, **kwargs):
            if isinstance(path_or_bytes, (str, os.PathLike)):
                path_or_bytes = materialize(os.fspath(path_or_bytes))
            super().__init__(path_or_bytes, *args, **kwargs)

    ort.InferenceSession = Session  # the remote code calls ort.InferenceSession(...) at load time


class Adapter(BaseAdapter):
    def load(self, model_ref, cfg):
        import onnxruntime as ort
        patch_onnxruntime(ort)
        print(f"[indic-conformer] onnxruntime {ort.__version__}, available providers: "
              f"{ort.get_available_providers()}", file=sys.stderr)
        self.model = AutoModel.from_pretrained(model_ref, trust_remote_code=True, revision=cfg.get("revision"))
        sessions = getattr(self.model, "models", {}) or {}
        used = {n: s.get_providers()[0] for n, s in sessions.items() if hasattr(s, "get_providers")}
        print(f"[indic-conformer] session providers: {used}", file=sys.stderr)
        self.device = "GPU" if any("CUDA" in p for p in used.values()) else "CPU"

    def transcribe(self, audio_path, lang=None, decoding="ctc", **options):
        need_lang(lang, "IndicConformer")
        wav = torch.from_numpy(read_wav(audio_path)).unsqueeze(0)  # (1, T) at 16 kHz
        return {"text": self.model(wav, lang, decoding or "ctc"), "device": self.device}

In [ ]:
%%writefile /content/stt_router/adapters/mms_adapter.py
"""Meta MMS (wav2vec2 CTC + per-language adapters). Licence: CC-BY-NC 4.0."""
import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC

from base import BaseAdapter, device, need_lang, read_wav

# Router uses ISO 639-1 where possible; MMS wants ISO 639-3.
ISO3 = {"hi": "hin", "bn": "ben", "ta": "tam", "te": "tel", "mr": "mar", "gu": "guj",
        "kn": "kan", "ml": "mal", "pa": "pan", "or": "ory", "ur": "urd", "as": "asm",
        "ne": "npi", "en": "eng"}


class Adapter(BaseAdapter):
    def load(self, model_ref, cfg):
        self.dev = device()
        rev = cfg.get("revision")
        self.processor = AutoProcessor.from_pretrained(model_ref, revision=rev)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_ref, revision=rev).to(self.dev).eval()
        self.current = None

    def _use(self, lang):
        code = ISO3.get(lang, lang)
        if code != self.current:
            self.processor.tokenizer.set_target_lang(code)
            self.model.load_adapter(code)  # small download on first use of a language
            self.current = code

    def transcribe(self, audio_path, lang=None, **options):
        need_lang(lang, "MMS")
        self._use(lang)
        inputs = self.processor(read_wav(audio_path), sampling_rate=16000, return_tensors="pt").to(self.dev)
        with torch.inference_mode():
            ids = self.model(**inputs).logits.argmax(dim=-1)[0]
        return {"text": self.processor.decode(ids)}

## 5 · Worker, router and setup manager
`worker.py` runs inside a model venv. `router.py` runs in this notebook: it builds venvs, starts/stops workers, and handles timeouts.

In [ ]:
%%writefile /content/stt_router/worker.py
"""
Worker process. Runs INSIDE a model's own venv, never in the notebook kernel.

    <venv>/bin/python worker.py <adapter_file> <model_ref> <config_json>

Protocol (JSON lines):
    worker -> router   {"status": "ready", "load_seconds": 12.3}     once, after load
    worker -> router   {"status": "failed", "error": "..."}          if load fails
    router -> worker   {"audio": "/content/in/x.wav", "lang": "hi", "options": {...}}
    worker -> router   {"text": "...", "seconds": 2.4, ...}
    worker -> router   {"error": "..."}                              request failed, worker stays up
    router -> worker   {"cmd": "shutdown"}
"""
import importlib.util
import json
import os
import sys
import time
import traceback

# Model libraries love printing to stdout (progress bars, warnings), which would
# corrupt the protocol. Keep a private copy of the real stdout for protocol
# messages, then point fd 1 (and sys.stdout) at stderr, which goes to the log file.
_proto = os.fdopen(os.dup(1), "w", buffering=1, encoding="utf-8")
os.dup2(2, 1)
sys.stdout = sys.stderr


def send(obj):
    _proto.write(json.dumps(obj, ensure_ascii=False) + "\n")
    _proto.flush()


def load_adapter(path):
    sys.path.insert(0, os.path.dirname(os.path.abspath(path)))  # so adapters can `from base import ...`
    spec = importlib.util.spec_from_file_location("stt_adapter", path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.Adapter()


def main():
    adapter_path, model_ref, cfg = sys.argv[1], sys.argv[2], json.loads(sys.argv[3])
    t0 = time.time()
    try:
        adapter = load_adapter(adapter_path)
        adapter.load(model_ref, cfg)
    except BaseException as e:  # noqa: BLE001 - report anything, including SystemExit from libs
        traceback.print_exc()
        send({"status": "failed", "error": f"{type(e).__name__}: {e}"})
        return 1
    send({"status": "ready", "load_seconds": round(time.time() - t0, 2)})

    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            req = json.loads(line)
            if req.get("cmd") == "shutdown":
                break
            if req.get("cmd") == "ping":
                send({"status": "pong"})
                continue
            t = time.time()
            out = adapter.transcribe(req["audio"], req.get("lang"), **(req.get("options") or {}))
            out["seconds"] = round(time.time() - t, 3)
            send(out)
        except Exception as e:  # noqa: BLE001
            traceback.print_exc()
            send({"error": f"{type(e).__name__}: {e}"})
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/stt_router/router.py
"""
Router + setup manager. Runs in the notebook kernel (main process).
Never imports torch / transformers - every model runs in its own worker process.
"""
import hashlib
import json
import os
import queue
import shlex
import shutil
import signal
import subprocess
import sys
import threading
import time
from pathlib import Path

import yaml

ROOT = Path(os.environ.get("STT_ROOT", "/content/stt_router"))     # registry, worker, adapters
ENVS = Path(os.environ.get("STT_ENVS", "/content/envs"))            # venvs (fast local disk)
MODELS = Path(os.environ.get("STT_MODELS", "/content/models"))      # cloned GitHub code
INBOX = Path(os.environ.get("STT_INBOX", "/content/in"))            # converted audio
LOGS = ROOT / "logs"

READY_TIMEOUT = int(os.environ.get("STT_READY_TIMEOUT", 1800))   # first load can download GBs
REQUEST_TIMEOUT = int(os.environ.get("STT_REQUEST_TIMEOUT", 600))

for _p in (ENVS, MODELS, INBOX, LOGS):
    _p.mkdir(parents=True, exist_ok=True)


def load_registry():
    return yaml.safe_load((ROOT / "registry.yaml").read_text())["models"]


def log_tail(name, n=40):
    p = LOGS / f"{name}.log"
    return "\n".join(p.read_text(errors="replace").splitlines()[-n:]) if p.exists() else "(no log yet)"


# ------------------------------------------------------------------ weight cache
# Weights are LOADED from the local HF cache (fast, reliable). Google Drive only keeps a
# backup: one .tar per repo, written after a model has loaded successfully and restored
# before the next load on a fresh runtime. Reading big safetensors files straight from the
# Drive mount can return stale/partial bytes ("header too large"), so we never do that.

def hf_home():
    return Path(os.environ.get("HF_HOME", "/content/hf_cache"))


def drive_cache():
    p = os.environ.get("STT_DRIVE_CACHE", "")
    return Path(p) if p else None


def repo_cache_dir(repo):
    return hf_home() / "hub" / ("models--" + repo.replace("/", "--"))


def repo_tar(repo):
    d = drive_cache()
    return d / ("models--" + repo.replace("/", "--") + ".tar") if d else None


_backup_lock = threading.Lock()


def restore_from_drive(repo):
    """Unpack the Drive backup into the local cache if we don't have the repo locally yet."""
    tar, local = repo_tar(repo), repo_cache_dir(repo)
    if not tar or not tar.exists() or local.exists():
        return False
    local.parent.mkdir(parents=True, exist_ok=True)
    tmp = local.parent / (local.name + ".restoring")
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir()
    r = subprocess.run(["tar", "-xf", str(tar), "-C", str(tmp)], capture_output=True, text=True)
    if r.returncode or not (tmp / local.name).exists():
        shutil.rmtree(tmp, ignore_errors=True)
        tar.unlink(missing_ok=True)  # unreadable backup: drop it, the model will re-download
        return False
    (tmp / local.name).rename(local)
    tmp.rmdir()
    return True


def backup_to_drive(repo, log_path):
    """Tar the local cache of a repo to Drive (in the background). Atomic: tmp name, then rename."""
    tar, local = repo_tar(repo), repo_cache_dir(repo)
    if not tar or tar.exists() or not local.exists():
        return

    def work():
        with _backup_lock, open(log_path, "a") as log:
            try:
                tar.parent.mkdir(parents=True, exist_ok=True)
                size = sum(f.stat().st_size for f in local.rglob("*") if f.is_file() and not f.is_symlink())
                free = shutil.disk_usage(tar.parent).free
                if free < size * 1.1:
                    log.write(f"[backup] skipped {repo}: Drive has {free/1e9:.1f} GB free, need {size/1e9:.1f} GB\n")
                    return
                tmp = tar.with_suffix(".tar.partial")
                r = subprocess.run(["tar", "-cf", str(tmp), "--exclude=*.incomplete", "-C",
                                    str(local.parent), local.name], capture_output=True, text=True)
                if r.returncode:
                    tmp.unlink(missing_ok=True)
                    log.write(f"[backup] tar failed for {repo}: {r.stderr}\n")
                    return
                tmp.rename(tar)
                log.write(f"[backup] saved {repo} to Drive ({size/1e9:.2f} GB)\n")
            except Exception as e:  # noqa: BLE001 - backup is best effort
                log.write(f"[backup] failed for {repo}: {e}\n")

    threading.Thread(target=work, daemon=True).start()


def purge_cache(repo):
    """Delete a repo's local cache AND its Drive backup (used when files are corrupted)."""
    shutil.rmtree(repo_cache_dir(repo), ignore_errors=True)
    shutil.rmtree(hf_home() / "onnx_real", ignore_errors=True)  # hard links made by ONNX adapters
    t = repo_tar(repo)
    if t:
        t.unlink(missing_ok=True)


# Error text that means "the cached weight files are damaged", not "the code is wrong".
CORRUPT_MARKERS = ("header too large", "safetensorerror", "incomplete metadata", "invalid load key",
                   "pytorchstreamreader failed", "unexpected eof", "file is not a zip file",
                   "consistency check failed", "error while deserializing")


def looks_corrupted(text):
    t = text.lower()
    return any(m in t for m in CORRUPT_MARKERS)


def _run(cmd, log_path):
    """Run a shell command, append its output to the model's log, raise on failure."""
    with open(log_path, "a") as f:
        f.write(f"\n$ {cmd}\n")
        f.flush()
        rc = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT).returncode
    if rc:
        raise RuntimeError(f"Setup step failed (exit {rc}): {cmd}\n--- log tail ---\n{log_tail(log_path.stem, 25)}")


# ------------------------------------------------------------------ setup manager
class SetupManager:
    def fingerprint(self, spec):
        key = json.dumps([spec.get("deps", []), spec.get("isolated_torch", False), sys.version], sort_keys=True)
        return hashlib.sha1(key.encode()).hexdigest()[:12]

    def is_ready(self, name, spec):
        f = ENVS / name / ".ready"
        return f.exists() and f.read_text().strip() == self.fingerprint(spec)

    def ensure(self, name, spec):
        """Generator: yields progress strings. Final value via .result -> (python_path, model_ref)."""
        env, log = ENVS / name, LOGS / f"{name}.log"
        py = env / "bin" / "python"

        if not self.is_ready(name, spec):
            shutil.rmtree(env, ignore_errors=True)
            deps = " ".join(shlex.quote(d) for d in spec.get("deps", []))
            if spec.get("isolated_torch"):
                # Own torch: uv is fast and picks the matching CUDA wheel.
                yield f"Creating isolated venv for {name} ..."
                _run(f"uv venv {env} --python {shlex.quote(sys.executable)} -q", log)
                yield "Installing packages incl. torch (uv) - this can take a few minutes ..."
                if deps:
                    _run(f"uv pip install --python {py} --torch-backend auto {deps}", log)
            else:
                # Share Colab's torch. pip (not uv) because uv ignores packages that come
                # from --system-site-packages and would re-download torch.
                yield f"Creating venv for {name} (shares Colab's torch) ..."
                _run(f"uv venv {env} --seed --system-site-packages --python {shlex.quote(sys.executable)} -q", log)
                if deps:
                    yield f"Installing: {', '.join(spec['deps'])} ..."
                    _run(f"{py} -m pip install -q --disable-pip-version-check {deps}", log)
            (env / ".ready").write_text(self.fingerprint(spec))
        else:
            yield f"{name}: venv already set up"

        model_ref = spec["repo"]
        if spec.get("source") == "github":
            dest = MODELS / name
            if not (dest / ".git").exists():
                yield f"Cloning {spec['repo']} ..."
                _run(f"git clone --depth 1 {shlex.quote(spec['repo'])} {dest}", log)
                if spec.get("revision"):
                    _run(f"cd {dest} && git fetch --depth 1 origin {shlex.quote(spec['revision'])} "
                         f"&& git checkout FETCH_HEAD", log)
            model_ref = str(dest)
        else:
            if drive_cache() and not repo_cache_dir(spec["repo"]).exists() and repo_tar(spec["repo"]).exists():
                yield f"Restoring {spec['repo']} weights from the Drive backup ..."
                if not restore_from_drive(spec["repo"]):
                    yield "Drive backup was unreadable - it will be downloaded again."
        if spec.get("source") != "github" and spec.get("prefetch"):
            yield f"Checking / downloading {spec['repo']} from Hugging Face ..."
            from huggingface_hub import snapshot_download
            try:
                model_ref = snapshot_download(spec["repo"], revision=spec.get("revision"),
                                              allow_patterns=spec.get("allow_patterns"),
                                              token=os.environ.get("HF_TOKEN") or None)
            except Exception as e:
                hint = (" This model is gated: accept its terms on the HF page and set HF_TOKEN."
                        if spec.get("gated") else "")
                raise RuntimeError(f"Download failed for {spec['repo']}: {e}.{hint}") from e

        self.result = (str(py), model_ref)


# ------------------------------------------------------------------ worker
class Worker:
    def __init__(self, name, python, model_ref, spec):
        self.name = name
        self.lock = threading.Lock()
        self.q = queue.Queue()
        self.log = open(LOGS / f"{name}.log", "a", buffering=1)
        self.log.write(f"\n===== worker start {time.ctime()} =====\n")
        cfg = {"revision": spec.get("revision"), "options": spec.get("options", {})}
        adapter = ROOT / "adapters" / f"{spec['adapter']}.py"
        env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"}
        self.proc = subprocess.Popen(
            [python, "-u", str(ROOT / "worker.py"), str(adapter), model_ref, json.dumps(cfg)],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=self.log,
            text=True, bufsize=1, env=env, start_new_session=True,  # own process group -> clean kill
        )
        threading.Thread(target=self._pump, daemon=True).start()

    def _pump(self):
        try:
            for line in self.proc.stdout:
                try:
                    self.q.put(json.loads(line))
                except json.JSONDecodeError:
                    self.log.write(f"[non-protocol stdout] {line}")
        except (ValueError, OSError):  # pipe closed by stop()
            pass
        self.q.put({"status": "exited", "code": self.proc.wait()})

    def _next(self, timeout):
        try:
            return self.q.get(timeout=timeout)
        except queue.Empty:
            raise TimeoutError(f"{self.name}: no answer within {timeout}s") from None

    def wait_ready(self, timeout=READY_TIMEOUT):
        msg = self._next(timeout)
        if msg.get("status") != "ready":
            err = msg.get("error") or f"worker exited with code {msg.get('code')}"
            raise RuntimeError(f"{self.name} failed to load: {err}\n--- log tail ---\n{log_tail(self.name, 25)}")
        return msg

    def alive(self):
        return self.proc.poll() is None

    def request(self, payload, timeout=REQUEST_TIMEOUT):
        with self.lock:
            while not self.q.empty():  # drop stale messages
                self.q.get_nowait()
            self.proc.stdin.write(json.dumps(payload, ensure_ascii=False) + "\n")
            self.proc.stdin.flush()
            msg = self._next(timeout)
        if msg.get("status") == "exited":
            raise RuntimeError(f"{self.name} crashed (exit {msg.get('code')})\n{log_tail(self.name, 25)}")
        return msg

    def stop(self, timeout=20):
        if self.alive():
            try:
                self.proc.stdin.write(json.dumps({"cmd": "shutdown"}) + "\n")
                self.proc.stdin.flush()
                self.proc.wait(timeout=timeout)
            except Exception:  # noqa: BLE001
                pass
        if self.alive():
            try:
                os.killpg(self.proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            self.proc.wait()
        for pipe in (self.proc.stdin, self.proc.stdout):
            try:
                pipe.close()
            except Exception:  # noqa: BLE001
                pass
        self.log.write(f"===== worker stopped {time.ctime()} =====\n")
        self.log.close()


def gpu_pids():
    if not shutil.which("nvidia-smi"):
        return set()
    out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout
    return {int(x) for x in out.split() if x.strip().isdigit()}


def gpu_memory():
    if not shutil.which("nvidia-smi"):
        return "no GPU (CPU only)"
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    return f"GPU memory used / total: {out}"


# ------------------------------------------------------------------ router
class Router:
    def __init__(self, registry=None):
        self.registry = registry or load_registry()
        self.setup = SetupManager()
        self.active = None  # (name, Worker)
        self.switch_lock = threading.Lock()

    def reload_registry(self):
        self.registry = load_registry()

    @property
    def active_name(self):
        return self.active[0] if self.active and self.active[1].alive() else None

    def stop_active(self):
        if not self.active:
            return
        name, w = self.active
        pid = w.proc.pid
        w.stop()
        self.active = None
        for _ in range(30):  # wait until the GPU has really let go of the process
            if pid not in gpu_pids():
                break
            time.sleep(0.5)

    def activate(self, name):
        """Generator yielding status lines. Stops the old worker BEFORE starting the new one."""
        with self.switch_lock:
            if self.active_name == name:
                yield f"{name} is already loaded."
                return
            if name not in self.registry:
                raise KeyError(f"{name} is not in registry.yaml")
            spec = self.registry[name]
            if self.active:
                yield f"Stopping {self.active[0]} and freeing the GPU ..."
                self.stop_active()
            for attempt in (1, 2):
                yield from self.setup.ensure(name, spec)
                python, model_ref = self.setup.result
                yield f"Starting worker and loading {name} (the first time includes downloading weights) ..."
                w = Worker(name, python, model_ref, spec)
                try:
                    ready = w.wait_ready()
                    break
                except Exception as e:
                    w.stop()
                    if attempt == 1 and spec.get("source") != "github" and looks_corrupted(str(e)):
                        yield ("Cached weight files are damaged (" + str(e).splitlines()[0][:120] +
                               ") - deleting the cache and downloading again ...")
                        purge_cache(spec["repo"])
                        continue
                    raise
            self.active = (name, w)
            yield f"{name} ready - loaded in {ready['load_seconds']} s. {gpu_memory()}"
            if spec.get("source") != "github" and drive_cache() and not repo_tar(spec["repo"]).exists():
                backup_to_drive(spec["repo"], LOGS / f"{name}.log")
                yield "Saving a backup of the weights to Drive in the background."

    def transcribe(self, name, wav_path, lang=None, options=None):
        if self.active_name != name:
            for _ in self.activate(name):
                pass
        _, w = self.active
        try:
            msg = w.request({"audio": str(wav_path), "lang": lang, "options": options or {}})
        except TimeoutError:
            # A stuck worker cannot be trusted: kill it; the next request restarts it.
            self.stop_active()
            raise
        if "error" in msg:
            raise RuntimeError(f"{name}: {msg['error']}")
        return msg

    def shutdown(self):
        self.stop_active()


def set_hf_token(token):
    """Validate a Hugging Face token and make it active for downloads and new workers.
    Kept in memory only (environment of this runtime), never written to disk."""
    token = (token or "").strip()
    if not token:
        os.environ.pop("HF_TOKEN", None)
        return None
    from huggingface_hub import whoami
    info = whoami(token=token)  # raises if the token is invalid
    os.environ["HF_TOKEN"] = token
    return info.get("name", "?")


def gated_access(repo):
    """'granted', 'no access', or a short reason. Uses the token currently in HF_TOKEN."""
    try:
        from huggingface_hub import HfApi
        HfApi().auth_check(repo, token=os.environ.get("HF_TOKEN") or None)
        return "granted"
    except AttributeError:
        return "unknown (update huggingface_hub)"
    except Exception as e:  # noqa: BLE001
        name = type(e).__name__
        if "Gated" in name or "401" in str(e) or "403" in str(e):
            return "no access - accept the terms on the model page"
        return f"unknown ({name})"


def to_wav16k(src, tag="clip"):
    """Convert any upload to 16 kHz mono 16-bit WAV so every worker gets identical input."""
    dst = INBOX / f"{tag}_{int(time.time() * 1000)}.wav"
    r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src),
                        "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(dst)],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"ffmpeg could not read the audio: {r.stderr.strip()}")
    return dst

In [ ]:
%%writefile /content/stt_router/ui.py
import gradio as gr
import jiwer

import router as R

router = R.Router()


def reg():
    return router.registry


def model_choices():
    return [(spec.get("display", name), name) for name, spec in reg().items()]


def option_of(name):
    """(key, choices) of the model's first per-request UI option, or (None, [])."""
    opts = reg()[name].get("ui_options") or {}
    return next(iter(opts.items()), (None, []))


def model_info(name):
    s = reg()[name]
    ready = "set up" if router.setup.is_ready(name, s) else "not set up yet (first load installs it)"
    gated = " | **gated**: accept terms on the HF page + HF_TOKEN" if s.get("gated") else ""
    return (f"**Repo:** [{s['repo']}](https://huggingface.co/{s['repo']}) | venv: {ready}"
            f"{' | own torch' if s.get('isolated_torch') else ''}{gated}")


def on_model_change(name):
    langs = reg()[name]["languages"]
    key, choices = option_of(name)
    return (gr.update(choices=langs, value=langs[0]),
            gr.update(choices=choices, value=choices[0] if choices else None,
                      visible=bool(choices), label=f"Option: {key}" if key else "Option"),
            model_info(name))


def load_model(name):
    lines = []
    try:
        for msg in router.activate(name):
            lines.append(msg)
            yield "\n".join(lines), model_info(name)
    except Exception as e:  # noqa: BLE001
        lines.append(f"ERROR: {e}")
        yield "\n".join(lines), model_info(name)


def transcribe(name, audio_path, lang, opt_value):
    if not audio_path:
        yield "", "", "Upload or record some audio first."
        return
    status = []
    try:
        if router.active_name != name:
            for msg in router.activate(name):
                status.append(msg)
                yield "", "", "\n".join(status)
        wav = R.to_wav16k(audio_path)
        status.append("Transcribing ...")
        yield "", "", "\n".join(status)
        key, _ = option_of(name)
        out = router.transcribe(name, wav, lang, {key: opt_value} if key and opt_value else {})
        meta = f"**{out['seconds']} s** on `{name}`"
        if out.get("device"):
            meta += f" | ran on {out['device']}"
        if out.get("detected_lang"):
            meta += f" | detected language: `{out['detected_lang']}`"
        status.append("Done.")
        yield out["text"], meta, "\n".join(status)
    except Exception as e:  # noqa: BLE001
        status.append(f"ERROR: {e}")
        yield "", "", "\n".join(status)


def compare(names, audio_path, lang, reference):
    if not names or not audio_path:
        yield [], "Pick at least one model and some audio."
        return
    wav = R.to_wav16k(audio_path, "compare")
    rows = []
    for name in names:
        langs = reg()[name]["languages"]
        use = lang if lang in langs else None
        if use is None and "auto" not in langs:
            rows.append([name, f"(skipped: does not list language '{lang}')", None, None])
            yield rows, f"Skipped {name}"
            continue
        yield rows, f"Running {name} (loads the model if needed) ..."
        try:
            out = router.transcribe(name, wav, use)
            wer = round(jiwer.wer(reference.strip(), out["text"].strip()), 3) if reference.strip() else None
            rows.append([name, out["text"], out["seconds"], wer])
        except Exception as e:  # noqa: BLE001
            rows.append([name, f"ERROR: {str(e).splitlines()[0]}", None, None])
        yield rows, f"Finished {name}"
    yield rows, "All done. Time = transcription only (model load excluded)."


def token_status():
    import os
    if not os.environ.get("HF_TOKEN"):
        return "No token set - public models work, gated ones will fail to download."
    lines = ["Token active. Gated models:"]
    for name, spec in reg().items():
        if spec.get("gated"):
            lines.append(f"- `{name}`: {R.gated_access(spec['repo'])} "
                         f"([page](https://huggingface.co/{spec['repo']}))")
    return "\n".join(lines)


def use_token(token):
    try:
        user = R.set_hf_token(token)
    except Exception as e:  # noqa: BLE001
        return gr.update(), f"**Token rejected:** {type(e).__name__}: {str(e).splitlines()[0][:200]}"
    if user is None:
        return gr.update(value=""), "Token cleared."
    note = ("\n\nThe loaded model keeps running with the old token; models you load from now on use this one."
            if router.active_name else "")
    return gr.update(value=""), f"Signed in as **{user}**.\n\n{token_status()}{note}"


def status_text():
    lines = [f"Active model: {router.active_name or 'none'}", R.gpu_memory(), "", "Venvs:"]
    for name, spec in reg().items():
        lines.append(f"  {'[ready]' if router.setup.is_ready(name, spec) else '[ -- ]'} {name}")
    return "\n".join(lines)


def stop_model():
    router.stop_active()
    return status_text()


def reload_registry():
    router.reload_registry()
    choices = model_choices()
    return (gr.update(choices=choices, value=choices[0][1]),
            gr.update(choices=choices), status_text())


ALL_LANGS = sorted({l for s in reg().values() for l in s["languages"]})
first = model_choices()[0][1]

with gr.Blocks(title="STT Model Router") as demo:
    gr.Markdown("# Speech-to-Text Model Router\nPick a model, load it, then upload or record audio. "
                "Each model runs in its own venv and process; switching stops the old one first.")
    with gr.Accordion("Hugging Face token (needed for gated models)", open=False):
        with gr.Row():
            hf_token = gr.Textbox(type="password", show_label=False, scale=4,
                                  placeholder="hf_... (read access is enough; leave empty + Use to clear)")
            token_btn = gr.Button("Use token", scale=1)
        token_md = gr.Markdown(token_status())
        token_btn.click(use_token, hf_token, [hf_token, token_md])
        hf_token.submit(use_token, hf_token, [hf_token, token_md])
    with gr.Tab("Transcribe"):
        with gr.Row():
            with gr.Column(scale=1):
                model = gr.Dropdown(model_choices(), value=first, label="Model")
                info = gr.Markdown(model_info(first))
                load_btn = gr.Button("Load model")
                lang = gr.Dropdown(reg()[first]["languages"], value=reg()[first]["languages"][0],
                                   label="Language")
                _k, _c = option_of(first)
                opt = gr.Dropdown(_c, value=_c[0] if _c else None, visible=bool(_c), label=f"Option: {_k}")
                audio = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Audio")
                go = gr.Button("Transcribe", variant="primary")
            with gr.Column(scale=1):
                text = gr.Textbox(label="Transcript", lines=8)
                meta = gr.Markdown()
                status = gr.Textbox(label="Status", lines=6)
                with gr.Accordion("Model log (last 40 lines)", open=False):
                    log_box = gr.Textbox(lines=14, show_label=False)
                    gr.Button("Refresh log").click(lambda n: R.log_tail(n), model, log_box)
        model.change(on_model_change, model, [lang, opt, info])
        load_btn.click(load_model, model, [status, info])
        go.click(transcribe, [model, audio, lang, opt], [text, meta, status])

    with gr.Tab("Compare"):
        gr.Markdown("Runs one clip through several models, one after another (only one model "
                    "is on the GPU at a time). Paste a reference transcript to get WER.")
        c_models = gr.CheckboxGroup(model_choices(), label="Models")
        with gr.Row():
            c_audio = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Audio")
            with gr.Column():
                c_lang = gr.Dropdown(ALL_LANGS, value="hi" if "hi" in ALL_LANGS else ALL_LANGS[0],
                                     label="Language")
                c_ref = gr.Textbox(label="Reference transcript (optional, for WER)", lines=3)
        c_go = gr.Button("Run comparison", variant="primary")
        c_table = gr.Dataframe(headers=["model", "transcript", "seconds", "WER"], wrap=True)
        c_status = gr.Markdown()
        c_go.click(compare, [c_models, c_audio, c_lang, c_ref], [c_table, c_status])

    with gr.Tab("System"):
        sys_box = gr.Textbox(status_text(), lines=12, label="Status")
        with gr.Row():
            gr.Button("Refresh").click(status_text, None, sys_box)
            gr.Button("Stop active model (free GPU)").click(stop_model, None, sys_box)
            reload_b = gr.Button("Reload registry.yaml")
        reload_b.click(reload_registry, None, [model, c_models, sys_box])

## 6 · Build the app
Re-running this cell stops any running worker first, so you never end up with two models on the GPU.

In [ ]:
import sys, importlib
sys.path.insert(0, "/content/stt_router")

if "ui" in sys.modules:                       # re-run: clean up the previous app
    try:
        ui.router.shutdown(); ui.demo.close()
    except Exception as e:
        print("cleanup:", e)
import router as R; importlib.reload(R)
import ui;          importlib.reload(ui)
print("Models in registry:", ", ".join(ui.router.registry))

### (Optional) Quick test without the UI
Upload an audio file in the Files panel (📁, left), set `AUDIO`, and run. Good for checking a new model.

In [ ]:
AUDIO = "/content/sample.wav"           # any format ffmpeg can read
MODEL = "whisper-small-vaani-hindi"     # a key from registry.yaml
LANG  = "hi"

for msg in ui.router.activate(MODEL):
    print(msg)
out = ui.router.transcribe(MODEL, R.to_wav16k(AUDIO), LANG)
print("\n", out)

## 7 · Launch the UI
**Token box** (top): paste your Hugging Face token for gated models.  
**Transcribe** tab: pick a model → *Load model* (first time installs + downloads) → audio → *Transcribe*.  
**Compare** tab: run one clip through several models; paste a reference text for WER.  
**System** tab: GPU memory, which venvs are ready, stop the active model, reload the registry.

In [ ]:
ui.demo.queue().launch(share=SHARE, debug=False)

## 8 · Shut down
Stops the worker (frees the GPU) and closes the UI.

In [ ]:
ui.router.shutdown()
ui.demo.close()
print("Stopped.")

## Notes and troubleshooting

**Adding a model (example: another Whisper fine-tune)**
```yaml
  whisper-medium-vaani-kannada:
    display: "Whisper medium | Vaani Kannada"
    source: hf
    repo: ARTPARK-IISc/whisper-medium-vaani-kannada
    revision: main
    prefetch: false
    deps: [transformers, soundfile]
    isolated_torch: false
    languages: [kn]
    adapter: whisper_adapter      # reuse an existing adapter
```
Then press *Reload registry.yaml* in the System tab. A model with a new API needs a new file in `adapters/` with `load()` and `transcribe()` (copy `whisper_adapter.py` as a starting point).

**A model fails to load** → open *Model log* in the Transcribe tab, or read `/content/stt_router/logs/<model>.log`. Every pip install and the worker's full traceback go there.

**Gated model: 401/403** → accept the terms on the model's HF page, then paste the token in the UI's token box and check that the model shows *granted*.

**`SafetensorError: header too large` or similar** → the cached weight file is damaged. The router now detects this, deletes that model's cache (local and Drive backup), downloads again and retries once. If it still fails, check the model log.

**Library conflicts** → pin versions in that model's `deps` (e.g. `transformers==4.46.3`). Changing `deps` rebuilds that venv automatically on the next load. If a model needs a different torch, set `isolated_torch: true` and add `torch==<version>` to `deps`.

**Why pip and not uv for shared venvs?** `uv pip install` does not see packages inherited through `--system-site-packages`, so it would download a fresh multi-GB torch into every venv. The venv's own pip does see Colab's torch. uv is still used to create venvs, and to install isolated-torch models.

**ONNX models (IndicConformer): `External data path escapes model directory`** → newer onnxruntime rejects ONNX weight files reached through the Hugging Face cache's symlinks. The registry pins onnxruntime 1.20.x as the model card asks, and the adapter also opens the model from a folder of hard-linked real files, so it works either way. The Transcribe tab shows whether it ran on GPU or CPU; if it says CPU, check the model log for onnxruntime's CUDA provider messages.

**After a runtime reset** → `/content` is wiped, so venvs are rebuilt (a few minutes, mostly pip) and weights are restored from the Drive backup (`MyDrive/stt_router/weights_backup`). The backup is written in the background after a model's first successful load, and skipped if Drive does not have enough free space (see the model log).

**GPU memory** → only one model is loaded at a time. On a free T4 (15 GB) all models in the starting registry fit one at a time.

**Licences** → MMS-1B-all is CC-BY-NC (non-commercial). Indic-Transcribe-Flex needs Bodhan AI's written permission to host as a service for others; testing it yourself is fine.